# Notebook 4: Batch Inference with run_batch() on SPCS

This notebook demonstrates **distributed batch inference** using `ModelVersion.run_batch()`
which runs inference on Snowpark Container Services (SPCS) compute pools.

Key advantages of `run_batch()` over `mv.run()`:
- Distributed execution across multiple nodes via Ray
- Scales to millions/billions of records
- Runs on dedicated SPCS compute pools (CPU or GPU)
- Output written directly to a Snowflake stage

**Prerequisites**: Run Notebooks 01-03 first.

In [ ]:
import json
import pandas as pd
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.model._client.model.batch_inference_specs import OutputSpec, SaveMode, JobSpec

print("Libraries loaded.")

## 1. Connect to Snowflake

In [ ]:
session = Session.builder.config("connection_name", "DEMO").create()
session.sql("USE ROLE ACCOUNTADMIN").collect()
session.sql("USE DATABASE HEALTHCARE_ML").collect()
session.sql("USE SCHEMA INFERENCE").collect()
session.sql("USE WAREHOUSE HEALTHCARE_ML_WH").collect()

print(f"Connected: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")

## 2. Load Model from Registry

In [ ]:
registry = Registry(
    session=session,
    database_name="HEALTHCARE_ML",
    schema_name="MODEL_REGISTRY"
)

mv = registry.get_model("READMISSION_PREDICTOR").version("V1")
print(f"Model loaded: READMISSION_PREDICTOR V1")
print(f"Comment: {mv.comment}")

## 3. Prepare Input Data as Snowpark DataFrame

`run_batch()` takes a Snowpark DataFrame as input. We'll use the Feature View
dynamic table which contains all 33 features for every admission.

In [ ]:
# Load feature columns from metadata
with open('artifacts/model_metadata.json', 'r') as f:
    metadata = json.load(f)
FEATURE_COLUMNS = metadata['feature_columns']

# Read from the Feature View dynamic table (all admissions with computed features)
feature_df = session.table("HEALTHCARE_ML.FEATURE_STORE.PATIENT_CLINICAL_FEATURES$V1")
print(f"Feature View rows: {feature_df.count()}")

# Select only the feature columns needed for prediction + identifiers
input_df = feature_df.select(
    ["PATIENT_ID", "ADMISSION_ID"] + FEATURE_COLUMNS
)
print(f"Input DataFrame columns: {len(input_df.columns)}")
input_df.show(5)

## 4. Create Output Stage

In [ ]:
# Ensure the output stage exists
session.sql("""
    CREATE STAGE IF NOT EXISTS HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT
    COMMENT = 'Output location for batch inference results'
""").collect()
print("Output stage ready: @HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT")

## 5. Run Distributed Batch Inference

This calls `mv.run_batch()` which:
1. Builds a Docker image with model dependencies (first run only)
2. Launches a Ray cluster on the SPCS compute pool
3. Distributes inference across nodes
4. Writes results to the specified stage location

Using `DEMO_POOL_CPU` (CPU_X64_S, 1-10 nodes).

In [ ]:
# Prepare features-only DataFrame for the model (no ID columns)
features_only_df = feature_df.select(FEATURE_COLUMNS)
print(f"Batch inference input: {features_only_df.count()} rows, {len(FEATURE_COLUMNS)} features")

# Launch distributed batch inference
print("\nStarting run_batch() on DEMO_POOL_CPU...")
print("(First run may take several minutes for Docker image build)")

job = mv.run_batch(
    compute_pool="DEMO_POOL_CPU",
    X=features_only_df,
    output_spec=OutputSpec(
        stage_location="@HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT/readmission_predictions/",
        mode=SaveMode.OVERWRITE
    ),
    job_spec=JobSpec(
        function_name="predict_proba"
    )
)

print(f"Job submitted: {job}")

In [ ]:
# Monitor job status
import time

print("Monitoring job status...")
while True:
    status = job.status
    print(f"  Status: {status}")
    if status in ('DONE', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)

if status == 'DONE':
    print("\nBatch inference completed successfully!")
else:
    print(f"\nJob ended with status: {status}")
    try:
        print(f"Logs: {job.logs()}")
    except Exception as e:
        print(f"Could not retrieve logs: {e}")

## 6. Retrieve and Analyze Results

In [ ]:
# List output files
output_files = session.sql("""
    LIST @HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT/readmission_predictions/
""").collect()
print(f"Output files: {len(output_files)}")
for f in output_files:
    print(f"  {f['name']} ({f['size']} bytes)")

In [ ]:
# Load results into a Snowpark DataFrame
results_df = session.read.parquet(
    "@HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT/readmission_predictions/"
)
print(f"Results: {results_df.count()} rows")
print(f"Columns: {results_df.columns}")
results_df.show(10)

In [ ]:
# Save results as a table for downstream analysis
results_df.write.save_as_table(
    "HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS",
    mode="overwrite"
)

# Verify
count = session.sql("SELECT COUNT(*) AS CNT FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS").collect()
print(f"Batch predictions saved: {count[0]['CNT']} rows")

In [ ]:
# Quick analysis of predictions
analysis = session.sql("""
    SELECT 
        COUNT(*) AS total_predictions,
        AVG("output_feature_1") AS avg_readmission_probability,
        COUNT(CASE WHEN "output_feature_1" > 0.5 THEN 1 END) AS high_risk_count,
        COUNT(CASE WHEN "output_feature_1" <= 0.5 THEN 1 END) AS low_risk_count
    FROM HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS
""").collect()

print(f"\n=== Batch Prediction Analysis ===")
print(f"Total predictions: {analysis[0]['TOTAL_PREDICTIONS']}")
print(f"Avg readmission probability: {analysis[0]['AVG_READMISSION_PROBABILITY']:.4f}")
print(f"High risk (>50%): {analysis[0]['HIGH_RISK_COUNT']}")
print(f"Low risk (<=50%): {analysis[0]['LOW_RISK_COUNT']}")

## 7. Verification Summary

In [ ]:
print("="*60)
print("BATCH INFERENCE VERIFICATION")
print("="*60)
print(f"\nCompute Pool: DEMO_POOL_CPU (CPU_X64_S)")
print(f"Model: READMISSION_PREDICTOR V1")
print(f"Method: run_batch() - distributed via Ray on SPCS")
print(f"Output: @HEALTHCARE_ML.INFERENCE.BATCH_OUTPUT")
print(f"Results Table: HEALTHCARE_ML.INFERENCE.BATCH_PREDICTIONS")
print(f"\nBatch inference: VERIFIED")
print(f"\nReady for Notebook 05: Real-time inference with Online Feature Store")

session.close()
print("\nSession closed.")